# A3 — Retriever Evaluation: Query Rerouting Strategies

Compare four retrieval strategies on multi-hop questions:

| # | Strategy | Description |
|---|----------|-------------|
| 1 | **No Rerouting** (baseline) | Top-N results by distance across all collections |
| 2 | **Query Rerouting** | Filter results to classifier-predicted collections |
| 3 | **Forced Equal (all collections)** | Equal distribution from every collection (no classifier) |
| 4 | **Query Rerouting + Forced Multicollection** | Equal distribution within classifier-predicted collections |

Each strategy is evaluated globally **and** broken down by source pair (newspaper-only vs. newspaper + débats).

In [1]:
import json
import os
import sys

import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from scripts.config import (
    COHERE_API_KEY, EMBEDDINGS_1887_COHERE,
    DEBATS_COLLECTION_NAME, QUESTIONS_COLLECTION_NAME,
    DEFAULT_K_VALUES, DEFAULT_CLASS_TO_COLLECTIONS,
)
from scripts.data_loading import (
    load_newspaper_collections, load_questions_df,
    prepare_universal_df, embed_for_classification,
)
from scripts.utils.chroma_utils import load_chroma_collection
from scripts.utils.models import load_classifier_model
from scripts.retrieval_utils import (
    RAGRetriever_all_collections_with_classification,
    evaluate_and_print_metrics,
    retrieve_for_strategy,
    apply_no_rerouting,
    apply_query_rerouting,
    apply_forced_multicollection,
    apply_forced_equal_from_all_collections,
)

print("Imports OK")

Imports OK


In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────────

N_RESULTS = 10                              # Documents to keep after strategy application
K_VALUES = DEFAULT_K_VALUES                 # [3, 5, 10]
CLASS_TO_COLLECTIONS = DEFAULT_CLASS_TO_COLLECTIONS

BASE_DIR = os.getcwd() if os.path.basename(os.getcwd()) != "notebooks" else os.path.join(os.getcwd(), "..")
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "RETRIEVER_results_MH_query_rerouting")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output dir: {OUTPUT_DIR}")
print(f"K values: {K_VALUES}")

Output dir: c:\Users\plathoon\Desktop\Aurelien\debattre-generate-questions\data\RETRIEVER_results_MH_query_rerouting
K values: [3, 5, 10]


## Load Data

In [3]:
# ── Load collections ───────────────────────────────────────────────────────────────

newspaper_collections = load_newspaper_collections(EMBEDDINGS_1887_COHERE)

debats_collection = load_chroma_collection(
    EMBEDDINGS_1887_COHERE, api_key=COHERE_API_KEY, collection_name=DEBATS_COLLECTION_NAME
)
print(f"  Loaded: Les Débats ({debats_collection.count()} docs)")

collections_dict = {**newspaper_collections, "Les Débats": debats_collection}

  Loaded: Le Gaulois (78 docs)
  Loaded: L'Intransigeant (79 docs)
  Loaded: Les Débats (5103 docs)


In [4]:
# ── Load multi-hop questions ───────────────────────────────────────────────────────

questions_df = load_questions_df(EMBEDDINGS_1887_COHERE)
print(f"Columns: {questions_df.columns.tolist()}")

Loaded 885 questions
Columns: ['question', 'source_1', 'source_2', 'gold_ids', 'id_1', 'question_index', 'unique_id', 'id_2', 'question_type']


In [5]:
# ── Classifier + universal retriever ───────────────────────────────────────────────

model = load_classifier_model(classifier_type="lr")

retriever_universal = RAGRetriever_all_collections_with_classification(
    collections_dict=collections_dict,
    classifier_model=model,
    embedding_function=embed_for_classification,
    n_results_per_collection=15,
)

print(
    f"Retriever: {retriever_universal.n_results_per_collection} docs/collection "
    f"x {len(collections_dict)} collections"
)

c:\Users\plathoon\Desktop\Aurelien\debattre-generate-questions\data\models\lr_source_classifier.pkl
  Loaded Logistic Regression classifier from: c:\Users\plathoon\Desktop\Aurelien\debattre-generate-questions\data\models\lr_source_classifier.pkl
Retriever: 15 docs/collection x 3 collections


In [6]:
# ── Run / load universal retrieval ─────────────────────────────────────────────────
# Uncomment to re-run retrieval (expensive — Cohere API calls):
# from scripts.run_multi_hop_rag_experiment import perform_multi_hop_retrieval
# retrieval_results_file = os.path.join(OUTPUT_DIR, "MH_retrieval_results_universal_wCosine.jsonl")
# perform_multi_hop_retrieval(questions_df, retriever_universal, output_file=retrieval_results_file)

retrieval_results_file = os.path.join(OUTPUT_DIR, "MH_retrieval_results_universal_wCosine.jsonl")
df_universal = prepare_universal_df(retrieval_results_file)

Loaded 885 questions with pre-computed retrieval and classification


## Evaluation Helper

In [7]:
def evaluate_strategy(df_all, strategy_name, k_values=K_VALUES):
    """Evaluate a strategy globally and broken down by source pair."""
    print(f"\n{'='*80}")
    print(f"  {strategy_name}")
    print(f"{'='*80}")

    # Global evaluation
    results = evaluate_and_print_metrics(
        df_all, dataset_name=f"{strategy_name} (all)", k_values=k_values
    )

    # Newspaper-only pairs (source_2 != Les Débats)
    df_newspapers = df_all[df_all["source_2"] != "Les Débats"]
    if len(df_newspapers) > 0:
        evaluate_and_print_metrics(
            df_newspapers, dataset_name=f"{strategy_name} (newspapers only)", k_values=k_values
        )

    # Newspaper + Débats pairs
    df_debats = df_all[df_all["source_2"] == "Les Débats"]
    if len(df_debats) > 0:
        evaluate_and_print_metrics(
            df_debats, dataset_name=f"{strategy_name} (with Débats)", k_values=k_values
        )

    return results

## Strategy 1 — No Query Rerouting (Baseline)
Top-N results by distance across all collections, no filtering.

In [8]:
df_no_rerouting = apply_no_rerouting(df_universal, n_results=N_RESULTS)
results_no_rerouting = evaluate_strategy(df_no_rerouting, "No Rerouting (Baseline)")


  No Rerouting (Baseline)

EVALUATION RESULTS FOR: NO REROUTING (BASELINE) (ALL)

Computing recall metrics...


100%|██████████| 885/885 [00:00<00:00, 13945.35it/s]



Recall Metrics for 885 questions:
  recall@3: 0.362
  recall@5: 0.444
  recall@10: 0.549


Computing Accuracy: 100%|██████████| 885/885 [00:00<00:00, 13834.13it/s]



Accuracy Metrics for 885 questions:
  accuracy@3: 0.156
  accuracy@5: 0.251
  accuracy@10: 0.377


...


Recall Metrics for 571 questions:
  recall@3: 0.482
  recall@5: 0.580
  recall@10: 0.690


Computing Accuracy: 100%|██████████| 571/571 [00:00<00:00, 6943.59it/s]



Accuracy Metrics for 571 questions:
  accuracy@3: 0.236
  accuracy@5: 0.364
  accuracy@10: 0.520


Computing MRR: 100%|██████████| 571/571 [00:00<00:00, 5446.07it/s]


MRR Metrics for 571 questions:
  mrr@3: 0.347
  mrr@5: 0.370
  mrr@10: 0.385


## Strategy 2 — Query Rerouting
Filter results to classifier-predicted collections only.

In [9]:
df_rerouting = apply_query_rerouting(df_universal, CLASS_TO_COLLECTIONS, n_results=N_RESULTS)
results_rerouting = evaluate_strategy(df_rerouting, "Query Rerouting")


  Query Rerouting

EVALUATION RESULTS FOR: QUERY REROUTING (ALL)

Computing recall metrics...


100%|██████████| 885/885 [00:00<00:00, 12544.68it/s]



Recall Metrics for 885 questions:
  recall@3: 0.482
  recall@5: 0.575
  recall@10: 0.687


Computing Accuracy: 100%|██████████| 885/885 [00:00<00:00, 16979.90it/s]



Accuracy Metrics for 885 questions:
  accuracy@3: 0.254
  accuracy@5: 0.368
  accuracy@10: 0.527


...


Recall Metrics for 571 questions:
  recall@3: 0.482
  recall@5: 0.580
  recall@10: 0.690


Computing Accuracy: 100%|██████████| 571/571 [00:00<00:00, 8526.50it/s]



Accuracy Metrics for 571 questions:
  accuracy@3: 0.236
  accuracy@5: 0.364
  accuracy@10: 0.520


Computing MRR: 100%|██████████| 571/571 [00:00<00:00, 5793.81it/s]


MRR Metrics for 571 questions:
  mrr@3: 0.347
  mrr@5: 0.370
  mrr@10: 0.385


## Strategy 3 — Forced Equal Distribution (All Collections)
Equal number of documents from every collection regardless of classifier prediction.

In [12]:
df_forced_equal = apply_forced_equal_from_all_collections(df_universal, n_results=5)
results_forced_equal = evaluate_strategy(df_forced_equal, "Forced Equal (All Collections)")


  Forced Equal (All Collections)

EVALUATION RESULTS FOR: FORCED EQUAL (ALL COLLECTIONS) (ALL)

Computing recall metrics...


100%|██████████| 885/885 [00:00<00:00, 13452.69it/s]



Recall Metrics for 885 questions:
  recall@3: 0.501
  recall@5: 0.577
  recall@10: 0.577


Computing Accuracy: 100%|██████████| 885/885 [00:00<00:00, 12476.21it/s]



Accuracy Metrics for 885 questions:
  accuracy@3: 0.271
  accuracy@5: 0.348
  accuracy@10: 0.348


...


Recall Metrics for 571 questions:
  recall@3: 0.563
  recall@5: 0.601
  recall@10: 0.601


Computing Accuracy: 100%|██████████| 571/571 [00:00<00:00, 11400.66it/s]



Accuracy Metrics for 571 questions:
  accuracy@3: 0.317
  accuracy@5: 0.347
  accuracy@10: 0.347


Computing MRR: 100%|██████████| 571/571 [00:00<00:00, 5170.65it/s]


MRR Metrics for 571 questions:
  mrr@3: 0.391
  mrr@5: 0.400
  mrr@10: 0.400


## Strategy 4 — Query Rerouting + Forced Multicollection
Equal distribution within classifier-predicted collections.

In [13]:
df_forced_multi = apply_forced_multicollection(df_universal, CLASS_TO_COLLECTIONS, n_results=N_RESULTS)
results_forced_multi = evaluate_strategy(df_forced_multi, "Rerouting + Forced Multicollection")


  Rerouting + Forced Multicollection

EVALUATION RESULTS FOR: REROUTING + FORCED MULTICOLLECTION (ALL)

Computing recall metrics...


100%|██████████| 885/885 [00:00<00:00, 17611.92it/s]



Recall Metrics for 885 questions:
  recall@3: 0.482
  recall@5: 0.637
  recall@10: 0.727


Computing Accuracy: 100%|██████████| 885/885 [00:00<00:00, 20429.17it/s]



Accuracy Metrics for 885 questions:
  accuracy@3: 0.254
  accuracy@5: 0.451
  accuracy@10: 0.573


...


Recall Metrics for 571 questions:
  recall@3: 0.482
  recall@5: 0.675
  recall@10: 0.744


Computing Accuracy: 100%|██████████| 571/571 [00:00<00:00, 9586.85it/s]



Accuracy Metrics for 571 questions:
  accuracy@3: 0.236
  accuracy@5: 0.492
  accuracy@10: 0.574


Computing MRR: 100%|██████████| 571/571 [00:00<00:00, 9428.07it/s]



MRR Metrics for 571 questions:
  mrr@3: 0.347
  mrr@5: 0.393
  mrr@10: 0.403


## Strategy Comparison

In [14]:
all_strategies = [
    ("No Rerouting",             results_no_rerouting),
    ("Query Rerouting",          results_rerouting),
    ("Forced Equal (All)",       results_forced_equal),
    ("Rerouting + Forced Multi", results_forced_multi),
]

rows = []
for name, res in all_strategies:
    row = {"Strategy": name}
    for metric in ["recall", "accuracy", "mrr"]:
        for k in K_VALUES:
            row[f"{metric}@{k}"] = np.mean(res[metric][f"{metric}@{k}"])
    rows.append(row)

comparison_df = pd.DataFrame(rows)
comparison_df.style.highlight_max(
    axis=0, subset=[c for c in comparison_df.columns if c != "Strategy"]
)

,Strategy,recall@3,recall@5,recall@10,accuracy@3,accuracy@5,accuracy@10,mrr@3,mrr@5,mrr@10
0,No Rerouting,0.361582,0.443503,0.549153,0.155932,0.250847,0.377401,0.260734,0.279548,0.293737
1,Query Rerouting,0.481921,0.575141,0.687006,0.254237,0.368362,0.526554,0.346704,0.368004,0.383139
2,Forced Equal (All),0.500565,0.576836,0.576836,0.271186,0.348023,0.348023,0.325706,0.343616,0.343616
3,Rerouting + Forced Multi,0.481921,0.636723,0.727119,0.254237,0.450847,0.572881,0.346704,0.382863,0.395115
